In [0]:
%run ../00_Utils/utils

## O objetivo aqui é entender o que temos de dados analitcos disponivel
- Temos dados de quais tipos? string, int, double, date...
- Temos dados duplicados?
    - Se sim, porque? Temos que remover ou normalizar?
- Qual relação entre as tabelas?

### Categorias

In [0]:
df_categorias = spark.table("workspace.lakehouse_panvel.bronze_categorias")

In [0]:
df_categorias.show(truncate=False)

## Filiais

In [0]:
df_filiais = spark.table("workspace.lakehouse_panvel.bronze_filiais")

In [0]:
df_filiais.printSchema()

In [0]:
verificar_duplicidade(df_filiais, ["id_filial"])
verificar_duplicidade(df_filiais, ["data_abertura"])
# verificar_duplicidade(df_filiais, ["regional"]) - 17

In [0]:
df_filiais.groupBy("tipo_loja", "uf").count().orderBy("count", ascending=False).display()

In [0]:
# df_filiais.where(F.col("cidade") == "Porto Alegre").display()
df_filiais.display()

In [0]:
df_filiais.select(
    F.min("data_abertura").alias("primeira_filial"),
    F.max("data_abertura").alias("ultima_filial")
).show()

## Produtos

In [0]:
df_produtos = spark.table("workspace.lakehouse_panvel.bronze_produtos")
df_produtos.printSchema()

In [0]:
display(df_produtos)

In [0]:
verificar_duplicidade(df_produtos, ["id_produto"])
verificar_duplicidade(df_produtos, ["sku"])
verificar_duplicidade(df_produtos, ["nome_produto"])

In [0]:
df_produtos.where(F.col("nome_produto") == "Mamadeira 300ml - Legrand").orderBy("id_produto").show(truncate=False)

In [0]:
df_produtos.groupBy("tipo_produto").count().orderBy("count", acending=False).show(truncate=False)

In [0]:
df_produtos.groupBy("controlado").count().orderBy("count", acending=False).show(truncate=False)

In [0]:
df_produtos.groupBy("id_categoria").count().orderBy("id_categoria", acending=False).show(truncate=False)

In [0]:
df_produtos.select(
    F.min("preco_tabela_atual").alias("valor_minimo"),
    F.max("preco_tabela_atual").alias("valor_maximo")
).show(truncate=False)

In [0]:
df_produtos.where(
    F.col("preco_tabela_atual").isNull() |
    (F.col("preco_tabela_atual") == 0)
).show(truncate=False)

## Vendas

In [0]:
df_vendas = spark.table("workspace.lakehouse_panvel.bronze_vendas")
df_vendas.printSchema()

In [0]:
display(df_vendas)

In [0]:
df_vendas.groupBy("id_filial").count().orderBy("id_filial", acending=False).show(truncate=False, n=999999)

In [0]:
df_vendas.agg(
    F.count("id_venda").alias("total_itens_venda"),
    F.countDistinct("id_venda").alias("total_vendas"),
    F.countDistinct("id_cliente_fidelidade").alias("clientes_fidelidade")
).show()

In [0]:
verificar_duplicidade(df_vendas, ["id_venda"])

In [0]:
(
    df_vendas
        .groupBy("id_venda")
        .agg(
            F.countDistinct("forma_pagamento").alias("qtd_status")
        )
        .filter(F.col("qtd_status") > 1)
        .show(truncate=False)
)

In [0]:
df_vendas_silver.groupBy("status_venda").count().orderBy("count", acending=False).show(truncate=False)


### Explorando SQL de possiveis perguntas humanas

In [0]:
%sql
SELECT f.regional,
       SUM(v.valor_total_venda) AS receita_liquida
FROM workspace.lakehouse_panvel.silver_vendas v
JOIN workspace.lakehouse_panvel.bronze_filiais f
  ON v.id_filial = f.id_filial
WHERE v.status_venda = 'Concluida'
  AND f.regional = 'Serra'
  AND date_trunc('month', v.data_venda) = date_trunc('month', DATE'2026-07-01')
GROUP BY f.regional;

In [0]:
%sql
WITH receita_mensal AS (
  SELECT v.id_filial, date_trunc('month', v.data_venda) AS mes,
         SUM(v.valor_total_venda) AS receita
  FROM workspace.lakehouse_panvel.silver_vendas v
  WHERE v.status_venda = 'Concluida'
  GROUP BY 1, 2
)
SELECT f.nome_filial,
       r_atual.receita  AS receita_mes_atual,
       r_anterior.receita AS receita_mes_anterior,
       (r_atual.receita - r_anterior.receita) / r_anterior.receita AS crescimento
FROM receita_mensal r_atual
JOIN receita_mensal r_anterior
  ON r_atual.id_filial = r_anterior.id_filial
  AND r_anterior.mes = add_months(r_atual.mes, -1)
JOIN workspace.lakehouse_panvel.bronze_filiais f
  ON r_atual.id_filial = f.id_filial
WHERE f.nome_filial = 'Panvel Porto Alegre'
  AND r_atual.mes = date_trunc('month', current_date());

In [0]:
%sql
SELECT p.tipo_categoria,
       SUM(i.valor_final_item) AS receita,
       SUM(i.valor_final_item) / SUM(SUM(i.valor_final_item)) OVER () AS participacao
FROM workspace.lakehouse_panvel.silver_itens_venda i
JOIN workspace.lakehouse_panvel.silver_produtos p
  ON i.id_produto = p.id_produto
JOIN workspace.lakehouse_panvel.silver_vendas v
  ON i.id_venda = v.id_venda
WHERE v.status_venda = 'Concluida'
GROUP BY p.tipo_categoria;

In [0]:
%sql
WITH receita_mensal AS (
  SELECT f.regional, date_trunc('month', v.data_venda) AS mes,
         SUM(v.valor_total_venda) AS receita
  FROM workspace.lakehouse_panvel.silver_vendas v
  JOIN workspace.lakehouse_panvel.bronze_filiais f
    ON v.id_filial = f.id_filial
  WHERE v.status_venda = 'Concluida'
  GROUP BY 1, 2
)
SELECT r_atual.regional,
       r_atual.receita   AS receita_mes_atual,
       r_ano_anterior.receita AS receita_mesmo_mes_ano_anterior,
       (r_atual.receita - r_ano_anterior.receita) / r_ano_anterior.receita AS evolucao
FROM receita_mensal r_atual
JOIN receita_mensal r_ano_anterior
  ON r_atual.regional = r_ano_anterior.regional
  AND r_ano_anterior.mes = add_months(r_atual.mes, -12)
WHERE r_atual.regional = 'Metro POA'
  AND r_atual.mes = date_trunc('month', current_date());